# SafeFundus — Notebook 07B: Normal-Aware JointEnergy Open-Set Scoring

## Purpose

Notebook 06C retrained the selected classifier as:

> **Swin-Tiny + ASL + explicit NORMAL output**

with 15 logits:

\[
[\text{NORMAL}, \text{14 known diseases}]
\]

The first 14-output JointEnergy experiment showed a structural bias: healthy normal images often looked unfamiliar because they had no positive disease output.

This notebook corrects that problem without retraining the network.

---

# Two energy formulations

## A. Standard 15-output JointEnergy — diagnostic baseline

\[
J_{ID}^{15}(x)
=
\sum_{c=1}^{15}
\log(1+\exp(z_c))
\]

and validation-normalized unknownness:

\[
J_U^{15}(x)
=
\frac{\mu_J-J_{ID}^{15}(x)}
{\sigma_J}
\]

This checks whether simply adding NORMAL to the original sum is sufficient.

---

## B. Proposed Normal-Aware Dual-Mode JointEnergy — primary

A familiar RFMiD image can belong to either of two familiar modes:

1. **Normal mode**
2. **Known-disease mode**

Normal evidence:

\[
E_N(x)=\log(1+\exp(z_{NORMAL}))
\]

Known-disease evidence:

\[
E_D(x)=
\sum_{c=1}^{14}
\log(1+\exp(z_c))
\]

Using familiar validation labels only:

\[
Z_N(x)
=
\frac{E_N(x)-\mu_N}{\sigma_N}
\]

where \(\mu_N,\sigma_N\) are fit from **validation Normal** images, and

\[
Z_D(x)
=
\frac{E_D(x)-\mu_D}{\sigma_D}
\]

where \(\mu_D,\sigma_D\) are fit from **validation Known_Only** images.

At inference, no ground-truth group is needed.

We ask whether the image matches **either** familiar mode:

\[
F_{NA}(x)=\max(Z_N(x), Z_D(x))
\]

and define unfamiliarity:

\[
J_{NA}(x)=-F_{NA}(x)
\]

Therefore:

> **larger \(J_{NA}\) = more unfamiliar / more referral-worthy**

---

# Referral threshold

The primary threshold is selected from the complete familiar validation set only:

\[
\tau_{NA}
=
Q_{0.90}(J_{NA}^{val})
\]

This targets approximately **10% familiar validation referral**.

No unknown-test case is used to tune:

- score normalization;
- referral threshold;
- disease thresholds;
- method choice.

Unknown test labels are used only for final evaluation.


In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import json
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
)

warnings.filterwarnings("ignore")

print("SafeFundus — 07B Normal-Aware JointEnergy")


SafeFundus — 07B Normal-Aware JointEnergy


# 2. Attach Notebook 06C Output

Attach the Kaggle output:

**`06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip`**

This notebook automatically searches `/kaggle/input`.

It supports either:

- the ZIP itself; or
- the extracted Kaggle notebook output directory.


In [2]:
# ============================================================
# 2. Locate / extract Notebook 06C output
# ============================================================

SOURCE_NAME = "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier"

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

NOTEBOOK_STEM = "07B_NormalAware_JointEnergy_OpenSet_Scoring"

OUTPUT_DIR = WORK_ROOT / NOTEBOOK_STEM
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
PER_SEED_DIR = OUTPUT_DIR / "per_seed"

for p in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIG_DIR,
    PER_SEED_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

source_root = None

zip_candidates = list(
    INPUT_ROOT.rglob(
        f"{SOURCE_NAME}.zip"
    )
)

if zip_candidates:
    source_zip = zip_candidates[0]

    extract_dir = (
        WORK_ROOT /
        "notebook06c_normalaware_extracted"
    )

    if extract_dir.exists():
        shutil.rmtree(
            extract_dir
        )

    extract_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        source_zip,
        "r",
    ) as zf:
        zf.extractall(
            extract_dir
        )

    source_root = extract_dir

    print("Found ZIP:")
    print(source_zip)

else:
    candidates = []

    for p in INPUT_ROOT.rglob(
        "validation_logits_predictions_15class.csv"
    ):
        if (
            "seed_42" in str(p)
            and "predictions" in str(p)
        ):
            candidates.append(
                p.parents[2]
            )

    if candidates:
        source_root = candidates[0]

        print(
            "Found extracted 06C output:"
        )
        print(source_root)

if source_root is None:
    raise FileNotFoundError(
        "Could not find the 06C normal-aware output. "
        "Attach 06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip "
        "to this Kaggle notebook."
    )

print("\nUsing source root:")
print(source_root)


Found ZIP:
/kaggle/input/notebooks/shouravdebnath/06c-swintiny-asl-normalaware-multiseed-classifier/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip

Using source root:
/kaggle/working/notebook06c_normalaware_extracted


# 3. Configuration


In [3]:
# ============================================================
# 3. Configuration
# ============================================================

CONFIG = {
    "seeds": [
        42,
        123,
        2026,
    ],

    "primary_val_false_referral_rate":
        0.10,

    "sensitivity_val_false_referral_rates": [
        0.05,
        0.10,
        0.15,
    ],

    "normal_class":
        "NORMAL",

    "known_diseases": [
        "DR", "MH", "ODC", "TSLN",
        "DN", "MYA", "ARMD", "BRVO",
        "ODP", "ODE", "LS", "RS",
        "CSR", "CRS",
    ],
}

NORMAL_CLASS = CONFIG[
    "normal_class"
]

KNOWN_DISEASES = CONFIG[
    "known_diseases"
]

MODEL_CLASSES = [
    NORMAL_CLASS,
    *KNOWN_DISEASES,
]

print(
    json.dumps(
        CONFIG,
        indent=2,
    )
)


{
  "seeds": [
    42,
    123,
    2026
  ],
  "primary_val_false_referral_rate": 0.1,
  "sensitivity_val_false_referral_rates": [
    0.05,
    0.1,
    0.15
  ],
  "normal_class": "NORMAL",
  "known_diseases": [
    "DR",
    "MH",
    "ODC",
    "TSLN",
    "DN",
    "MYA",
    "ARMD",
    "BRVO",
    "ODP",
    "ODE",
    "LS",
    "RS",
    "CSR",
    "CRS"
  ]
}


# 4. Load Saved 15-Class Logits and Validation Thresholds


In [4]:
# ============================================================
# 4. Data loading
# ============================================================

def load_seed_data(seed):

    seed_root = (
        source_root /
        f"seed_{seed}"
    )

    pred_root = (
        seed_root /
        "predictions"
    )

    table_root = (
        seed_root /
        "tables"
    )

    paths = {
        "validation":
            pred_root /
            "validation_logits_predictions_15class.csv",

        "familiar":
            pred_root /
            "familiar_test_logits_predictions_15class.csv",

        "unknown":
            pred_root /
            "unknown_test_logits_predictions_15class.csv",

        "mixed":
            pred_root /
            "mixed_test_logits_predictions_15class.csv",

        "intermediate":
            pred_root /
            "intermediate_test_logits_predictions_15class.csv",

        "thresholds":
            table_root /
            "validation_thresholds_15class.csv",
    }

    missing = [
        str(p)
        for p in paths.values()
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing required 06C files:\n"
            + "\n".join(missing)
        )

    result = {
        key: pd.read_csv(path)
        for key, path in paths.items()
    }

    expected_logits = [
        f"logit_{c}"
        for c in MODEL_CLASSES
    ]

    for split in [
        "validation",
        "familiar",
        "unknown",
        "mixed",
        "intermediate",
    ]:
        absent = [
            c for c in expected_logits
            if c not in result[split].columns
        ]

        if absent:
            raise ValueError(
                f"{split} missing logits: "
                f"{absent}"
            )

    threshold_map = dict(
        zip(
            result["thresholds"]["Class"],
            result["thresholds"]["Threshold"],
        )
    )

    result["threshold_map"] = (
        threshold_map
    )

    return result


for seed in CONFIG["seeds"]:

    data = load_seed_data(
        seed
    )

    print(
        f"Seed {seed}: "
        f"val={len(data['validation'])}, "
        f"familiar={len(data['familiar'])}, "
        f"unknown={len(data['unknown'])}, "
        f"mixed={len(data['mixed'])}"
    )


Seed 42: val=560, familiar=571, unknown=32, mixed=24
Seed 123: val=560, familiar=571, unknown=32, mixed=24
Seed 2026: val=560, familiar=571, unknown=32, mixed=24


# 5. Energy Functions


In [5]:
# ============================================================
# 5. Energy functions
# ============================================================

def softplus_np(x):
    return np.logaddexp(
        0.0,
        x,
    )


def standard_jointenergy_15(
    dataframe,
):
    cols = [
        f"logit_{c}"
        for c in MODEL_CLASSES
    ]

    logits = dataframe[
        cols
    ].to_numpy(
        dtype=np.float64
    )

    return softplus_np(
        logits
    ).sum(axis=1)


def energy_branches(
    dataframe,
):
    normal_logit = (
        dataframe[
            f"logit_{NORMAL_CLASS}"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    disease_cols = [
        f"logit_{c}"
        for c in KNOWN_DISEASES
    ]

    disease_logits = (
        dataframe[
            disease_cols
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    normal_evidence = (
        softplus_np(
            normal_logit
        )
    )

    disease_evidence = (
        softplus_np(
            disease_logits
        )
        .sum(axis=1)
    )

    return (
        normal_evidence,
        disease_evidence,
    )


def safe_std(x):
    value = float(
        np.std(
            x,
            ddof=0,
        )
    )

    if value < 1e-12:
        raise ValueError(
            "Standard deviation is too small."
        )

    return value


# 6. Fit Validation-Only Normal-Aware Energy

The branch normalizers are fit only once per seed using the familiar validation set.

### Normal branch

Fit only on validation images with:

`Group == "Normal"`

### Disease branch

Fit only on validation images with:

`Group == "Known_Only"`

At inference, the score itself does not use the true group.


In [6]:
# ============================================================
# 6. Fit / apply normal-aware energy
# ============================================================

def fit_normalaware_energy(
    validation_df,
):
    normal_evidence, disease_evidence = (
        energy_branches(
            validation_df
        )
    )

    groups = (
        validation_df[
            "Group"
        ]
        .astype(str)
        .to_numpy()
    )

    normal_mask = (
        groups
        == "Normal"
    )

    disease_mask = (
        groups
        == "Known_Only"
    )

    mu_normal = float(
        normal_evidence[
            normal_mask
        ].mean()
    )

    sd_normal = safe_std(
        normal_evidence[
            normal_mask
        ]
    )

    mu_disease = float(
        disease_evidence[
            disease_mask
        ].mean()
    )

    sd_disease = safe_std(
        disease_evidence[
            disease_mask
        ]
    )

    return {
        "mu_normal":
            mu_normal,
        "sd_normal":
            sd_normal,
        "mu_disease":
            mu_disease,
        "sd_disease":
            sd_disease,
    }


def normalaware_unknownness(
    dataframe,
    params,
):
    normal_evidence, disease_evidence = (
        energy_branches(
            dataframe
        )
    )

    z_normal = (
        normal_evidence
        - params["mu_normal"]
    ) / params["sd_normal"]

    z_disease = (
        disease_evidence
        - params["mu_disease"]
    ) / params["sd_disease"]

    # Familiar if either the normal branch
    # OR the known-disease branch matches strongly.
    familiar_evidence = np.maximum(
        z_normal,
        z_disease,
    )

    # Larger score = more unfamiliar.
    unknownness = (
        -familiar_evidence
    )

    return {
        "normal_evidence":
            normal_evidence,
        "disease_evidence":
            disease_evidence,
        "z_normal":
            z_normal,
        "z_disease":
            z_disease,
        "familiar_evidence":
            familiar_evidence,
        "unknownness":
            unknownness,
    }


# 7. Standard 15-Output JointEnergy Baseline

This is retained as an ablation.

Its normalization is fit on the complete familiar validation set.


In [7]:
# ============================================================
# 7. Standard 15-class JointEnergy
# ============================================================

def standard_15_unknownness(
    dataframe,
    mu,
    sigma,
):
    jid = standard_jointenergy_15(
        dataframe
    )

    ju = (
        mu - jid
    ) / sigma

    return jid, ju


# 8. OOD Evaluation Metrics

Unknown-only test images are the positive class.

We report:

- AUROC
- AUPR
- FPR@95% TPR

These use test labels only for evaluation, never threshold tuning.


In [8]:
# ============================================================
# 8. OOD metrics
# ============================================================

def fpr_at_95_tpr(
    y_true,
    scores,
):
    fpr, tpr, thresholds = (
        roc_curve(
            y_true,
            scores,
        )
    )

    valid = np.where(
        tpr >= 0.95
    )[0]

    if len(valid) == 0:
        return np.nan

    return float(
        np.min(
            fpr[valid]
        )
    )


def ood_metrics(
    familiar_scores,
    unknown_scores,
):
    y_true = np.concatenate([
        np.zeros(
            len(familiar_scores),
            dtype=int,
        ),
        np.ones(
            len(unknown_scores),
            dtype=int,
        ),
    ])

    scores = np.concatenate([
        familiar_scores,
        unknown_scores,
    ])

    return {
        "unknown_auroc":
            float(
                roc_auc_score(
                    y_true,
                    scores,
                )
            ),

        "unknown_aupr":
            float(
                average_precision_score(
                    y_true,
                    scores,
                )
            ),

        "fpr_at_95_tpr":
            fpr_at_95_tpr(
                y_true,
                scores,
            ),
    }


# 9. Safety Decision Helper

For comparability with earlier experiments, the primary rare-to-normal metric remains:

> unknown-only case is not referred **and** no known disease exceeds its disease threshold.

We also report a stricter secondary metric:

> unknown-only case is not referred, no disease is predicted, **and the explicit NORMAL head is positive**.

This second metric measures direct false reassurance by the new NORMAL head.


In [9]:
# ============================================================
# 9. Safety helper
# ============================================================

def unknown_closedset_decisions(
    unknown_df,
    threshold_map,
):
    disease_prob = np.stack(
        [
            unknown_df[
                f"prob_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in KNOWN_DISEASES
        ],
        axis=1,
    )

    disease_thresholds = np.asarray(
        [
            threshold_map[c]
            for c in KNOWN_DISEASES
        ],
        dtype=float,
    )

    disease_pred = (
        disease_prob
        >= disease_thresholds[
            None, :
        ]
    )

    any_disease = (
        disease_pred
        .any(axis=1)
    )

    no_disease = (
        ~any_disease
    )

    normal_probability = (
        unknown_df[
            f"prob_{NORMAL_CLASS}"
        ].to_numpy(
            dtype=float
        )
    )

    normal_threshold = float(
        threshold_map[
            NORMAL_CLASS
        ]
    )

    explicit_normal = (
        normal_probability
        >= normal_threshold
    )

    return {
        "any_disease":
            any_disease,
        "no_disease":
            no_disease,
        "explicit_normal":
            explicit_normal,
    }


# 10. Run Both Energy Methods Across All Three Seeds

The **Normal-Aware Dual-Mode score** is the proposed primary method.

The standard 15-output sum is an ablation.


In [10]:
# ============================================================
# 10. Multi-seed evaluation
# ============================================================

all_method_rows = []
threshold_sensitivity_rows = []

for seed in CONFIG["seeds"]:

    print("\n" + "#" * 80)
    print(
        f"NORMAL-AWARE JOINTENERGY — SEED {seed}"
    )
    print("#" * 80)

    data = load_seed_data(
        seed
    )

    val_df = data[
        "validation"
    ].copy()

    familiar_df = data[
        "familiar"
    ].copy()

    unknown_df = data[
        "unknown"
    ].copy()

    threshold_map = data[
        "threshold_map"
    ]

    # ========================================================
    # A. Standard 15-output JointEnergy
    # ========================================================

    val_jid15 = (
        standard_jointenergy_15(
            val_df
        )
    )

    mu15 = float(
        val_jid15.mean()
    )

    sd15 = safe_std(
        val_jid15
    )

    _, val_std15 = (
        standard_15_unknownness(
            val_df,
            mu15,
            sd15,
        )
    )

    _, familiar_std15 = (
        standard_15_unknownness(
            familiar_df,
            mu15,
            sd15,
        )
    )

    _, unknown_std15 = (
        standard_15_unknownness(
            unknown_df,
            mu15,
            sd15,
        )
    )

    # ========================================================
    # B. Proposed normal-aware dual-mode JointEnergy
    # ========================================================

    na_params = (
        fit_normalaware_energy(
            val_df
        )
    )

    val_na = (
        normalaware_unknownness(
            val_df,
            na_params,
        )
    )

    familiar_na = (
        normalaware_unknownness(
            familiar_df,
            na_params,
        )
    )

    unknown_na = (
        normalaware_unknownness(
            unknown_df,
            na_params,
        )
    )

    methods = {
        "Standard15_Sum": {
            "val_scores":
                val_std15,
            "familiar_scores":
                familiar_std15,
            "unknown_scores":
                unknown_std15,
        },

        "NormalAware_DualMode": {
            "val_scores":
                val_na["unknownness"],
            "familiar_scores":
                familiar_na["unknownness"],
            "unknown_scores":
                unknown_na["unknownness"],
        },
    }

    decisions = (
        unknown_closedset_decisions(
            unknown_df,
            threshold_map,
        )
    )

    familiar_groups = (
        familiar_df[
            "Group"
        ]
        .astype(str)
        .to_numpy()
    )

    normal_mask = (
        familiar_groups
        == "Normal"
    )

    known_mask = (
        familiar_groups
        == "Known_Only"
    )

    seed_out = (
        PER_SEED_DIR /
        f"seed_{seed}"
    )

    seed_out.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Save proposed branch values for later fusion analysis.
    proposed_val_df = pd.DataFrame({
        "ID":
            val_df["ID"],
        "Group":
            val_df["Group"],
        "Normal_Evidence":
            val_na["normal_evidence"],
        "Disease_Evidence":
            val_na["disease_evidence"],
        "Z_Normal":
            val_na["z_normal"],
        "Z_Disease":
            val_na["z_disease"],
        "Familiar_Evidence":
            val_na["familiar_evidence"],
        "NormalAware_Unknownness":
            val_na["unknownness"],
    })

    proposed_familiar_df = pd.DataFrame({
        "ID":
            familiar_df["ID"],
        "Group":
            familiar_df["Group"],
        "Normal_Evidence":
            familiar_na["normal_evidence"],
        "Disease_Evidence":
            familiar_na["disease_evidence"],
        "Z_Normal":
            familiar_na["z_normal"],
        "Z_Disease":
            familiar_na["z_disease"],
        "Familiar_Evidence":
            familiar_na["familiar_evidence"],
        "NormalAware_Unknownness":
            familiar_na["unknownness"],
    })

    proposed_unknown_df = pd.DataFrame({
        "ID":
            unknown_df["ID"],
        "Group":
            unknown_df["Group"],
        "Normal_Evidence":
            unknown_na["normal_evidence"],
        "Disease_Evidence":
            unknown_na["disease_evidence"],
        "Z_Normal":
            unknown_na["z_normal"],
        "Z_Disease":
            unknown_na["z_disease"],
        "Familiar_Evidence":
            unknown_na["familiar_evidence"],
        "NormalAware_Unknownness":
            unknown_na["unknownness"],
        "ClosedSet_No_Disease":
            decisions[
                "no_disease"
            ].astype(int),
        "Explicit_Normal_Predicted":
            decisions[
                "explicit_normal"
            ].astype(int),
    })

    proposed_val_df.to_csv(
        seed_out /
        "validation_normalaware_energy_scores.csv",
        index=False,
    )

    proposed_familiar_df.to_csv(
        seed_out /
        "familiar_test_normalaware_energy_scores.csv",
        index=False,
    )

    proposed_unknown_df.to_csv(
        seed_out /
        "unknown_test_normalaware_energy_scores.csv",
        index=False,
    )

    with open(
        seed_out /
        "normalaware_validation_parameters.json",
        "w",
    ) as f:
        json.dump(
            na_params,
            f,
            indent=2,
        )

    # ========================================================
    # Evaluate both methods at primary operating point
    # ========================================================

    for method_name, scores in (
        methods.items()
    ):

        val_scores = scores[
            "val_scores"
        ]

        familiar_scores = scores[
            "familiar_scores"
        ]

        unknown_scores = scores[
            "unknown_scores"
        ]

        target_frr = CONFIG[
            "primary_val_false_referral_rate"
        ]

        threshold = float(
            np.quantile(
                val_scores,
                1.0 - target_frr,
            )
        )

        val_ref = (
            val_scores
            > threshold
        )

        familiar_ref = (
            familiar_scores
            > threshold
        )

        unknown_ref = (
            unknown_scores
            > threshold
        )

        residual = (
            ~unknown_ref
        )

        # Legacy/comparable rare-to-normal definition.
        rare_to_normal = (
            residual
            & decisions[
                "no_disease"
            ]
        )

        # Strict direct NORMAL false reassurance.
        explicit_normal_false_reassurance = (
            residual
            & decisions[
                "no_disease"
            ]
            & decisions[
                "explicit_normal"
            ]
        )

        forced_known = (
            residual
            & decisions[
                "any_disease"
            ]
        )

        metric_row = {
            "seed":
                seed,
            "method":
                method_name,

            "threshold":
                threshold,

            "actual_val_false_referral_rate":
                float(
                    val_ref.mean()
                ),

            "familiar_test_false_referral_rate":
                float(
                    familiar_ref.mean()
                ),

            "normal_test_false_referral_rate":
                float(
                    familiar_ref[
                        normal_mask
                    ].mean()
                ),

            "known_only_test_false_referral_rate":
                float(
                    familiar_ref[
                        known_mask
                    ].mean()
                ),

            "unknown_referral_sensitivity":
                float(
                    unknown_ref.mean()
                ),

            "closedset_rare_to_normal_rate":
                float(
                    decisions[
                        "no_disease"
                    ].mean()
                ),

            "post_referral_rare_to_normal_rate":
                float(
                    rare_to_normal.mean()
                ),

            "post_referral_explicit_normal_false_reassurance":
                float(
                    explicit_normal_false_reassurance.mean()
                ),

            "post_referral_forced_known_rate":
                float(
                    forced_known.mean()
                ),
        }

        metric_row.update(
            ood_metrics(
                familiar_scores,
                unknown_scores,
            )
        )

        all_method_rows.append(
            metric_row
        )

        # ----------------------------------------------------
        # Threshold sensitivity only for proposed method.
        # ----------------------------------------------------

        if (
            method_name
            == "NormalAware_DualMode"
        ):
            for requested_frr in CONFIG[
                "sensitivity_val_false_referral_rates"
            ]:

                sensitivity_threshold = float(
                    np.quantile(
                        val_scores,
                        1.0 - requested_frr,
                    )
                )

                fam_ref_s = (
                    familiar_scores
                    > sensitivity_threshold
                )

                unk_ref_s = (
                    unknown_scores
                    > sensitivity_threshold
                )

                residual_s = (
                    ~unk_ref_s
                )

                rare_s = (
                    residual_s
                    & decisions[
                        "no_disease"
                    ]
                )

                strict_s = (
                    residual_s
                    & decisions[
                        "no_disease"
                    ]
                    & decisions[
                        "explicit_normal"
                    ]
                )

                threshold_sensitivity_rows.append({
                    "seed":
                        seed,

                    "requested_val_false_referral_rate":
                        requested_frr,

                    "threshold":
                        sensitivity_threshold,

                    "actual_val_false_referral_rate":
                        float(
                            (
                                val_scores
                                > sensitivity_threshold
                            ).mean()
                        ),

                    "familiar_test_false_referral_rate":
                        float(
                            fam_ref_s.mean()
                        ),

                    "normal_test_false_referral_rate":
                        float(
                            fam_ref_s[
                                normal_mask
                            ].mean()
                        ),

                    "known_only_test_false_referral_rate":
                        float(
                            fam_ref_s[
                                known_mask
                            ].mean()
                        ),

                    "unknown_referral_sensitivity":
                        float(
                            unk_ref_s.mean()
                        ),

                    "post_referral_rare_to_normal_rate":
                        float(
                            rare_s.mean()
                        ),

                    "post_referral_explicit_normal_false_reassurance":
                        float(
                            strict_s.mean()
                        ),
                })

    # ========================================================
    # Primary proposed score distribution
    # ========================================================

    proposed_threshold = float(
        np.quantile(
            val_na["unknownness"],
            1.0
            - CONFIG[
                "primary_val_false_referral_rate"
            ],
        )
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        familiar_na[
            "unknownness"
        ],
        bins=30,
        density=True,
        alpha=0.6,
        label="Familiar test",
    )

    plt.hist(
        unknown_na[
            "unknownness"
        ],
        bins=15,
        density=True,
        alpha=0.6,
        label="Unknown-only test",
    )

    plt.axvline(
        proposed_threshold,
        linestyle="--",
        label="Validation threshold",
    )

    plt.xlabel(
        "Normal-Aware JointEnergy Unknownness"
    )
    plt.ylabel(
        "Density"
    )
    plt.title(
        f"Normal-Aware JointEnergy — Seed {seed}"
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIG_DIR /
        f"normalaware_jointenergy_distribution_seed_{seed}.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

print("\nAll seeds evaluated.")



################################################################################
NORMAL-AWARE JOINTENERGY — SEED 42
################################################################################

################################################################################
NORMAL-AWARE JOINTENERGY — SEED 123
################################################################################

################################################################################
NORMAL-AWARE JOINTENERGY — SEED 2026
################################################################################

All seeds evaluated.


# 11. Compare Standard 15-Sum vs Proposed Normal-Aware JointEnergy


In [11]:
# ============================================================
# 11. Method comparison
# ============================================================

method_results_df = pd.DataFrame(
    all_method_rows
)

method_results_df.to_csv(
    TABLE_DIR /
    "jointenergy_method_results_by_seed.csv",
    index=False,
)

display(
    method_results_df
)

METRICS = [
    "familiar_test_false_referral_rate",
    "normal_test_false_referral_rate",
    "known_only_test_false_referral_rate",
    "unknown_referral_sensitivity",
    "closedset_rare_to_normal_rate",
    "post_referral_rare_to_normal_rate",
    "post_referral_explicit_normal_false_reassurance",
    "post_referral_forced_known_rate",
    "unknown_auroc",
    "unknown_aupr",
    "fpr_at_95_tpr",
]

aggregate_rows = []

for method_name in (
    method_results_df[
        "method"
    ].unique()
):
    subset = (
        method_results_df[
            method_results_df[
                "method"
            ] == method_name
        ]
    )

    for metric in METRICS:
        values = pd.to_numeric(
            subset[metric],
            errors="coerce",
        )

        aggregate_rows.append({
            "Method":
                method_name,
            "Metric":
                metric,
            "Mean":
                float(
                    values.mean()
                ),
            "SD":
                float(
                    values.std(
                        ddof=1
                    )
                ),
            "Min":
                float(
                    values.min()
                ),
            "Max":
                float(
                    values.max()
                ),
        })

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR /
    "jointenergy_method_comparison_mean_sd.csv",
    index=False,
)

display(
    aggregate_df
)

print("\n" + "=" * 80)
print(
    "PRIMARY NORMAL-AWARE DUAL-MODE SUMMARY"
)
print("=" * 80)

primary = (
    aggregate_df[
        aggregate_df["Method"]
        == "NormalAware_DualMode"
    ]
)

for _, row in primary.iterrows():
    print(
        f"{row['Metric']:52s}: "
        f"{row['Mean']:.6f} "
        f"± {row['SD']:.6f}"
    )


,seed,method,threshold,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,closedset_rare_to_normal_rate,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance,post_referral_forced_known_rate,unknown_auroc,unknown_aupr,fpr_at_95_tpr
0,42,Standard15_Sum,1.196646,0.1,0.110333,0.253731,0.066362,0.15625,0.21875,0.21875,0.09375,0.62500,0.645085,0.090595,0.793345
1,42,NormalAware_DualMode,0.880677,0.1,0.094571,0.044776,0.109840,0.28125,0.21875,0.18750,0.09375,0.53125,0.734895,0.136699,0.767075
2,123,Standard15_Sum,1.201725,0.1,0.068301,0.059701,0.070938,0.28125,0.28125,0.15625,0.09375,0.56250,0.740258,0.193453,0.651489
3,123,NormalAware_DualMode,1.012853,0.1,0.061296,0.022388,0.073227,0.25000,0.28125,0.15625,0.09375,0.59375,0.768717,0.198836,0.751313
4,2026,Standard15_Sum,1.223258,0.1,0.101576,0.179104,0.077803,0.12500,0.31250,0.25000,0.21875,0.62500,0.690346,0.101243,0.677758
5,2026,NormalAware_DualMode,1.009301,0.1,0.084063,0.082090,0.084668,0.21875,0.31250,0.18750,0.15625,0.59375,0.719626,0.137214,0.800350


,Method,Metric,Mean,SD,Min,Max
0,Standard15_Sum,familiar_test_false_referral_rate,0.093403,0.022176,0.068301,0.110333
1,Standard15_Sum,normal_test_false_referral_rate,0.164179,0.097872,0.059701,0.253731
2,Standard15_Sum,known_only_test_false_referral_rate,0.071701,0.005759,0.066362,0.077803
3,Standard15_Sum,unknown_referral_sensitivity,0.187500,0.082680,0.125000,0.281250
4,Standard15_Sum,closedset_rare_to_normal_rate,0.270833,0.047735,0.218750,0.312500
5,Standard15_Sum,post_referral_rare_to_normal_rate,0.208333,0.047735,0.156250,0.250000
6,Standard15_Sum,post_referral_explicit_normal_false_reassurance,0.135417,0.072169,0.093750,0.218750
7,Standard15_Sum,post_referral_forced_known_rate,0.604167,0.036084,0.562500,0.625000
8,Standard15_Sum,unknown_auroc,0.691897,0.047605,0.645085,0.740258
9,Standard15_Sum,unknown_aupr,0.128430,0.056562,0.090595,0.193453



PRIMARY NORMAL-AWARE DUAL-MODE SUMMARY
familiar_test_false_referral_rate                   : 0.079977 ± 0.017010
normal_test_false_referral_rate                     : 0.049751 ± 0.030160
known_only_test_false_referral_rate                 : 0.089245 ± 0.018731
unknown_referral_sensitivity                        : 0.250000 ± 0.031250
closedset_rare_to_normal_rate                       : 0.270833 ± 0.047735
post_referral_rare_to_normal_rate                   : 0.177083 ± 0.018042
post_referral_explicit_normal_false_reassurance     : 0.114583 ± 0.036084
post_referral_forced_known_rate                     : 0.572917 ± 0.036084
unknown_auroc                                       : 0.741079 ± 0.025123
unknown_aupr                                        : 0.157583 ± 0.035727
fpr_at_95_tpr                                       : 0.772913 ± 0.025034


# 12. Threshold Sensitivity — Proposed Method

The **10% validation false-referral target** remains the primary operating point.

The 5% and 15% points are sensitivity analysis only.


In [12]:
# ============================================================
# 12. Threshold sensitivity
# ============================================================

sensitivity_df = pd.DataFrame(
    threshold_sensitivity_rows
)

sensitivity_df.to_csv(
    TABLE_DIR /
    "normalaware_threshold_sensitivity_by_seed.csv",
    index=False,
)

display(
    sensitivity_df
)

sensitivity_aggregate = (
    sensitivity_df
    .groupby(
        "requested_val_false_referral_rate"
    )
    .agg(
        familiar_false_referral_mean=(
            "familiar_test_false_referral_rate",
            "mean",
        ),

        familiar_false_referral_sd=(
            "familiar_test_false_referral_rate",
            "std",
        ),

        normal_false_referral_mean=(
            "normal_test_false_referral_rate",
            "mean",
        ),

        known_false_referral_mean=(
            "known_only_test_false_referral_rate",
            "mean",
        ),

        unknown_referral_sensitivity_mean=(
            "unknown_referral_sensitivity",
            "mean",
        ),

        unknown_referral_sensitivity_sd=(
            "unknown_referral_sensitivity",
            "std",
        ),

        rare_to_normal_mean=(
            "post_referral_rare_to_normal_rate",
            "mean",
        ),

        strict_false_reassurance_mean=(
            "post_referral_explicit_normal_false_reassurance",
            "mean",
        ),
    )
    .reset_index()
)

sensitivity_aggregate.to_csv(
    TABLE_DIR /
    "normalaware_threshold_sensitivity_mean_sd.csv",
    index=False,
)

display(
    sensitivity_aggregate
)


,seed,requested_val_false_referral_rate,threshold,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance
0,42,0.05,1.082386,0.05,0.045534,0.014925,0.054920,0.18750,0.21875,0.09375
1,42,0.10,0.880677,0.10,0.094571,0.044776,0.109840,0.28125,0.18750,0.09375
2,42,0.15,0.725771,0.15,0.140105,0.082090,0.157895,0.37500,0.18750,0.09375
3,123,0.05,1.227483,0.05,0.031524,0.014925,0.036613,0.18750,0.18750,0.12500
4,123,0.10,1.012853,0.10,0.061296,0.022388,0.073227,0.25000,0.15625,0.09375
5,123,0.15,0.840589,0.15,0.099825,0.052239,0.114416,0.28125,0.15625,0.09375
6,2026,0.05,1.203433,0.05,0.029772,0.014925,0.034325,0.15625,0.21875,0.18750
7,2026,0.10,1.009301,0.10,0.084063,0.082090,0.084668,0.21875,0.18750,0.15625
8,2026,0.15,0.769560,0.15,0.161121,0.164179,0.160183,0.37500,0.12500,0.12500


,requested_val_false_referral_rate,familiar_false_referral_mean,familiar_false_referral_sd,normal_false_referral_mean,known_false_referral_mean,unknown_referral_sensitivity_mean,unknown_referral_sensitivity_sd,rare_to_normal_mean,strict_false_reassurance_mean
0,0.05,0.035610,0.008639,0.014925,0.041953,0.177083,0.018042,0.208333,0.135417
1,0.10,0.079977,0.017010,0.049751,0.089245,0.250000,0.031250,0.177083,0.114583
2,0.15,0.133684,0.031148,0.099502,0.144165,0.343750,0.054127,0.156250,0.104167


# 13. Safety Improvement Summary


In [13]:
# ============================================================
# 13. Primary safety summary
# ============================================================

primary_rows = (
    method_results_df[
        method_results_df["method"]
        == "NormalAware_DualMode"
    ]
)

baseline_rtn = float(
    primary_rows[
        "closedset_rare_to_normal_rate"
    ].mean()
)

post_rtn = float(
    primary_rows[
        "post_referral_rare_to_normal_rate"
    ].mean()
)

absolute_reduction = (
    baseline_rtn
    - post_rtn
)

relative_reduction = (
    absolute_reduction
    / baseline_rtn
    if baseline_rtn > 0
    else np.nan
)

summary = {
    "closedset_rare_to_normal_rate":
        baseline_rtn,

    "post_referral_rare_to_normal_rate":
        post_rtn,

    "absolute_rare_to_normal_reduction":
        absolute_reduction,

    "relative_rare_to_normal_reduction":
        relative_reduction,

    "mean_unknown_referral_sensitivity":
        float(
            primary_rows[
                "unknown_referral_sensitivity"
            ].mean()
        ),

    "mean_familiar_false_referral_rate":
        float(
            primary_rows[
                "familiar_test_false_referral_rate"
            ].mean()
        ),

    "mean_normal_false_referral_rate":
        float(
            primary_rows[
                "normal_test_false_referral_rate"
            ].mean()
        ),

    "mean_known_false_referral_rate":
        float(
            primary_rows[
                "known_only_test_false_referral_rate"
            ].mean()
        ),

    "mean_explicit_normal_false_reassurance":
        float(
            primary_rows[
                "post_referral_explicit_normal_false_reassurance"
            ].mean()
        ),
}

print(
    json.dumps(
        summary,
        indent=2,
    )
)

with open(
    TABLE_DIR /
    "normalaware_primary_safety_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )


{
  "closedset_rare_to_normal_rate": 0.2708333333333333,
  "post_referral_rare_to_normal_rate": 0.17708333333333334,
  "absolute_rare_to_normal_reduction": 0.09374999999999997,
  "relative_rare_to_normal_reduction": 0.3461538461538461,
  "mean_unknown_referral_sensitivity": 0.25,
  "mean_familiar_false_referral_rate": 0.07997664915353182,
  "mean_normal_false_referral_rate": 0.04975124378109452,
  "mean_known_false_referral_rate": 0.08924485125858124,
  "mean_explicit_normal_false_reassurance": 0.11458333333333333
}


# 14. Save Method Metadata


In [14]:
# ============================================================
# 14. Metadata
# ============================================================

metadata = {
    "project":
        "SafeFundus",

    "classifier":
        "Swin-Tiny + ASL + explicit NORMAL output",

    "primary_open_set_score":
        "Normal-Aware Dual-Mode JointEnergy",

    "normal_evidence":
        "softplus(NORMAL logit)",

    "disease_evidence":
        "sum softplus(14 disease logits)",

    "normal_branch_normalization":
        "validation Normal subgroup only",

    "disease_branch_normalization":
        "validation Known_Only subgroup only",

    "familiar_evidence":
        "max(z_normal, z_disease)",

    "unknownness":
        "-max(z_normal, z_disease)",

    "primary_referral_threshold":
        "90th percentile of familiar validation unknownness",

    "primary_target_validation_false_referral_rate":
        CONFIG[
            "primary_val_false_referral_rate"
        ],

    "unknown_test_used_for_tuning":
        False,
}

with open(
    OUTPUT_DIR /
    "normalaware_jointenergy_method_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print(
    json.dumps(
        metadata,
        indent=2,
    )
)


{
  "project": "SafeFundus",
  "classifier": "Swin-Tiny + ASL + explicit NORMAL output",
  "primary_open_set_score": "Normal-Aware Dual-Mode JointEnergy",
  "normal_evidence": "softplus(NORMAL logit)",
  "disease_evidence": "sum softplus(14 disease logits)",
  "normal_branch_normalization": "validation Normal subgroup only",
  "disease_branch_normalization": "validation Known_Only subgroup only",
  "familiar_evidence": "max(z_normal, z_disease)",
  "unknownness": "-max(z_normal, z_disease)",
  "primary_referral_threshold": "90th percentile of familiar validation unknownness",
  "primary_target_validation_false_referral_rate": 0.1,
  "unknown_test_used_for_tuning": false
}


# 15. Create One Complete Output ZIP

The notebook automatically creates:

**`07B_NormalAware_JointEnergy_OpenSet_Scoring.zip`**

The ZIP contains:

- per-image normal-aware energy scores;
- validation-only branch normalization parameters;
- standard 15-output JointEnergy ablation;
- proposed dual-mode JointEnergy results;
- false-referral rates;
- unknown referral sensitivity;
- rare-to-normal safety results;
- explicit-normal false reassurance;
- AUROC / AUPR / FPR@95TPR;
- 5% / 10% / 15% threshold sensitivity;
- score-distribution figures;
- multi-seed mean ± SD tables.


In [15]:
# ============================================================
# 15. Create complete ZIP
# ============================================================

ZIP_PATH = (
    WORK_ROOT /
    f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(
        ZIP_PATH.with_suffix("")
    ),
    format="zip",
    root_dir=OUTPUT_DIR,
)

size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("=" * 80)
print("COMPLETE OUTPUT ZIP CREATED")
print("=" * 80)
print(ZIP_PATH)
print(
    f"ZIP size: "
    f"{size_mb:.2f} MB"
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/07B_NormalAware_JointEnergy_OpenSet_Scoring.zip
ZIP size: 0.31 MB


# Next Step

Upload:

**`07B_NormalAware_JointEnergy_OpenSet_Scoring.zip`**

If the proposed normal-aware score behaves as expected, we then proceed to:

## Notebook 08 — MC Dropout Uncertainty

using the same frozen 06C Swin-Tiny + ASL + NORMAL checkpoints.

The later fusion will combine:

\[
\text{Normal-Aware JointEnergy}
+
\text{MC Dropout uncertainty}
\]

with all fusion normalization and thresholds fit using validation data only.
